# 00 — Session Setup

**Run cells top-to-bottom, one at a time. Wait for each to finish before running the next.**

---
### Before running: verify your Colab Secrets

Click the **🔑 key icon** in the left sidebar. You need these secrets with a **blue toggle** (Notebook access ON):
- `ANTHROPIC_API_KEY` — from console.anthropic.com
- `HF_TOKEN` — from huggingface.co/settings/tokens
- `GITHUB_PAT` — from github.com → Settings → Developer settings → Personal access tokens

If any toggle is grey, click it to turn it blue NOW before running anything.

---
### ⚠️ Change your branch name in Cell 3 before running it

In [1]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, ~15 GB VRAM
# If you see K80 or < 14 GB: Runtime → Disconnect and delete runtime → reconnect
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    icon = '✅' if vram >= 14 else '⚠️ '
    print(f'\n{icon} GPU: {name}  ({vram:.0f} GB VRAM)')
    if vram < 14:
        print('   You have a K80 (12 GB). Reconnect to get a T4 (16 GB).')
else:
    print('\n❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

Sun Oct  4 02:32:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A popup asks for permissions — click Allow on everything (normal Google behaviour).
from google.colab import drive
import os
drive.mount('/content/drive')
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Mount failed — run this cell again')

Mounted at /content/drive
✅ Drive mounted


In [11]:
# ── CELL 3: Clone or pull the GitHub repo ─────────────────────────────────────
#
# ⚠️  CHANGE BRANCH_NAME BEFORE RUNNING
#     Examples:
#       'advisor_colab_experiments'   ← advisor testing
#       'pair-1/smollm2-1.7b'         ← student pair 1
#       'main'                        ← read-only reference (do not push to main)

import os, subprocess, sys
from google.colab import userdata

BRANCH_NAME = 'BTT_SETUP_AC'          # ← CHANGE THIS
REPO_ORG    = 'Break-Through-Tech'
REPO_NAME   = 'Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation'
REPO_DIR    = '/content/project'   # repo root

# ── Load PAT ──────────────────────────────────────────────────────────────────
try:
    PAT = userdata.get('GITHUB_PAT')
    assert PAT, 'Secret is empty'
    print(f'✅ GITHUB_PAT loaded ({len(PAT)} chars)')
except Exception as e:
    print(f'❌ GITHUB_PAT: {e}')
    print('   Open 🔑 Secrets → add GITHUB_PAT → toggle Notebook access ON')
    raise SystemExit('Cannot clone without GITHUB_PAT')

REPO_URL = f'https://{PAT}@github.com/{REPO_ORG}/{REPO_NAME}.git'

def git(args, cwd=None, check=True):
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(f'❌ git error: {r.stderr.replace(PAT, "***").strip()}')
        raise RuntimeError(' '.join(args))
    return r.stdout.strip()

# ── Clone or pull ──────────────────────────────────────────────────────────────
if not os.path.exists(f'{REPO_DIR}/.git'):
    if os.path.exists(REPO_DIR):
        print('Removing broken directory ...')
        subprocess.run(['rm', '-rf', REPO_DIR])
    print(f'Cloning branch "{BRANCH_NAME}" ...')
    git(['git', 'clone', '-b', BRANCH_NAME, REPO_URL, REPO_DIR])
    print(f'✅ Cloned to {REPO_DIR}')
else:
    print(f'Pulling latest from "{BRANCH_NAME}" ...')
    git(['git', 'checkout', BRANCH_NAME], cwd=REPO_DIR)
    git(['git', 'pull', 'origin', BRANCH_NAME], cwd=REPO_DIR)
    print(f'✅ Up to date')

# ── Auto-detect where main.py lives (repo root or code/ subfolder) ────────────
if os.path.exists(f'{REPO_DIR}/main.py'):
    CODE_DIR = REPO_DIR
elif os.path.exists(f'{REPO_DIR}/code/main.py'):
    CODE_DIR = f'{REPO_DIR}/code'
else:
    CODE_DIR = None
    print('❌ Cannot find main.py — checked repo root and code/ subfolder')
    print(f'   Contents of {REPO_DIR}: {os.listdir(REPO_DIR)}')

if CODE_DIR:
    print(f'✅ Code directory: {CODE_DIR}')
    missing = [f for f in ['requirements.txt', 'requirements_colab.txt']
               if not os.path.exists(f'{CODE_DIR}/{f}')]
    if missing:
        print(f'⚠️  Missing in {CODE_DIR}: {missing}')
        print('   Push these files from your local machine, or run:')
        print('   !git -C /content/project fetch origin')
        print('   !git -C /content/project checkout origin/main -- code/requirements_colab.txt')
    else:
        print('✅ requirements.txt and requirements_colab.txt found')

    os.chdir(CODE_DIR)
    sys.path.insert(0, CODE_DIR)
    # Store CODE_DIR for other cells to use
    os.environ['SLM_CODE_DIR'] = CODE_DIR
    print(f'Working directory: {os.getcwd()}')

✅ GITHUB_PAT loaded (40 chars)
Pulling latest from "BTT_SETUP_AC" ...
✅ Up to date
✅ Code directory: /content/project/code
✅ requirements.txt and requirements_colab.txt found
Working directory: /content/project/code


In [4]:
# ── CELL 4: Install dependencies ──────────────────────────────────────────────
# Reads requirements files from the code directory found in Cell 3.
# Takes 3–4 minutes. Normal to see some warnings.
import os, subprocess, sys

CODE_DIR = os.environ.get('SLM_CODE_DIR', '/content/project/code')
print(f'Installing from: {CODE_DIR}')

def pip_install(filename):
    path = f'{CODE_DIR}/{filename}'
    if not os.path.exists(path):
        print(f'❌ {filename} not found at {path}')
        print('   Make sure Cell 3 ran successfully first.')
        return False
    print(f'\nInstalling {filename} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', path],
        capture_output=True, text=True
    )
    tail = (result.stdout + result.stderr).strip().split('\n')
    for line in tail[-5:]:
        if line.strip():
            print(f'  {line}')
    if result.returncode != 0:
        print(f'❌ pip failed for {filename}')
        return False
    print(f'✅ {filename} done')
    return True

ok1 = pip_install('requirements.txt')
ok2 = pip_install('requirements_colab.txt')
print('\n✅ All dependencies installed' if (ok1 and ok2) else '\n⚠️  Check errors above')

Installing from: /content/project/code

Installing requirements.txt ...
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 22.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.4 MB/s eta 0:00:00
✅ requirements.txt done

Installing requirements_colab.txt ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.8 MB/s eta 0:00:00
  ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
  gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
✅ requirements_colab.txt done

✅ All dependencies installed


In [5]:
# ── CELL 5: Drive folder structure and HuggingFace cache ─────────────────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
for d in [f'{DRIVE_ROOT}/data/raw', f'{DRIVE_ROOT}/data/processed',
          f'{DRIVE_ROOT}/data/checkpoints', f'{DRIVE_ROOT}/outputs',
          f'{DRIVE_ROOT}/hf_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT

print(f'✅ Drive folders ready under {DRIVE_ROOT}')
print(f'✅ HF model cache → {os.environ["HF_HOME"]}')

✅ Drive folders ready under /content/drive/MyDrive/slm-distillation
✅ HF model cache → /content/drive/MyDrive/slm-distillation/hf_cache


In [6]:
# ── CELL 6: Load API keys from Colab Secrets ──────────────────────────────────
import os
from google.colab import userdata

def load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'  ✅ {name}')
            return True
        print(f'  ⚠️  {name} is empty — {"add value in 🔑 Secrets" if required else "optional"}')
    except Exception:
        print(f'  ❌ {name} not found — {"open 🔑 Secrets and toggle Notebook access ON" if required else "optional"}')
    return not required

print('Loading secrets:')
all_ok = all([
    load_secret('ANTHROPIC_API_KEY', required=True),
    load_secret('HF_TOKEN',          required=True),
    load_secret('OPENAI_API_KEY',    required=False),
])
print('\n✅ Required secrets loaded' if all_ok else '\n❌ Fix missing secrets before running the pipeline')

Loading secrets:
  ✅ ANTHROPIC_API_KEY
  ✅ HF_TOKEN
  ❌ OPENAI_API_KEY not found — optional

✅ Required secrets loaded


In [12]:
# ── CELL 7: Verify everything is ready ───────────────────────────────────────
import os, torch

code_dir  = os.environ.get('SLM_CODE_DIR', '')
drive_ok  = os.path.exists('/content/drive/MyDrive')
gpu_ok    = torch.cuda.is_available()
vram_ok   = gpu_ok and torch.cuda.get_device_properties(0).total_memory > 14e9

checks = [
    ('T4 GPU (≥14 GB VRAM)',   vram_ok),
    ('Drive mounted',          drive_ok),
    ('Code directory found',   bool(code_dir) and os.path.exists(code_dir)),
    ('main.py present',        os.path.exists(f'{code_dir}/main.py') if code_dir else False),
    ('ANTHROPIC_API_KEY set',  'ANTHROPIC_API_KEY' in os.environ),
    ('HF_TOKEN set',           'HF_TOKEN' in os.environ),
    ('HF cache on Drive',      os.environ.get('HF_HOME','').startswith('/content/drive')),
]

print('Setup verification:')
print(f'  Code directory: {code_dir or "NOT SET"}')
print()
all_ok = True
for label, ok in checks:
    print(f'  {"✅" if ok else "❌"} {label}')
    if not ok:
        all_ok = False

print()
print('🚀 Ready! Scroll down to run the pipeline.' if all_ok else
      '⚠️  Fix ❌ items before running the pipeline.')

Setup verification:
  Code directory: /content/project/code

  ✅ T4 GPU (≥14 GB VRAM)
  ✅ Drive mounted
  ✅ Code directory found
  ✅ main.py present
  ✅ ANTHROPIC_API_KEY set
  ✅ HF_TOKEN set
  ✅ HF cache on Drive

🚀 Ready! Scroll down to run the pipeline.


---
## Run the pipeline

Choose one of the cells below. The `$SLM_CODE_DIR` variable is set by Cell 3.

In [13]:
# Add benchmark_latency.py to your branch as code/benchmark_latency.py first.
# Start a fresh Colab GPU runtime, then run your existing setup cells 1–7.
# Do NOT run the full-training or live-demo cells.
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata

CODE = Path(os.environ.get("SLM_CODE_DIR", "/content/project/code"))
DRIVE = Path(os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/slm-distillation"))
SCRIPT = CODE / "benchmark_latency.py"
assert SCRIPT.is_file(), "Add code/benchmark_latency.py to BTT_SETUP_AC, then rerun the clone/pull setup cell."
assert DRIVE.is_dir(), "Mount Google Drive first."

for name in ("HF_TOKEN", "ANTHROPIC_API_KEY"):
    if not os.environ.get(name):
        os.environ[name] = userdata.get(name)
    assert os.environ.get(name), f"Enable {name} in Colab Secrets."

# Optional: your effective GPU allocation cost in USD per hour.
# Leave None when unknown: GPU dollar costs will be N/A, not falsely $0.
# Explicit 0.0 represents zero incremental cash cost on a free allocation.
GPU_HOURLY_USD = None

command = [
    sys.executable, "-u", str(SCRIPT),
    "--config", str(CODE / "configs/phase1_config.yaml"),
    "--drive-root", str(DRIVE),
    "--base-model", "google/gemma-3-4b-it",
    "--compute-dtype", "float16",
    "--repeats", "3",
    "--warmups", "5",
    "--teacher-warmups", "1",
    "--teacher-model", "claude-haiku-4-5-20251001",
    "--teacher-input-usd-per-million", "1.0",
    "--teacher-output-usd-per-million", "5.0",
]
if GPU_HOURLY_USD is not None:
    command += ["--gpu-hourly-usd", str(GPU_HOURLY_USD)]

# Merge stdout/stderr so the underlying traceback is visible in Colab and saved.
LOG_DIR = DRIVE / "latency_benchmarks" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f") + ".log")

def run_visible(cmd):
    with LOG.open("a", encoding="utf-8") as log:
        with subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                for key in ("HF_TOKEN", "ANTHROPIC_API_KEY"):
                    if os.environ.get(key):
                        line = line.replace(os.environ[key], "[REDACTED]")
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Benchmark exited with code {status}. Read the traceback above. Full log: {LOG}")

run_visible(command + ["--dry-run"])
run_visible(command)
print(f"Full console log: {LOG}")

Workload: 50 requests; 10 clusters
SLM: 4 models × 50 × 3 measured calls
Teacher: 151 maximum paid calls; no automatic retries
Base: google/gemma-3-4b-it; GPU hourly rate: unknown
File checks passed; no GPU/model/API work performed.
Workload: 50 requests; 10 clusters
SLM: 4 models × 50 × 3 measured calls
Teacher: 151 maximum paid calls; no automatic retries
Base: google/gemma-3-4b-it; GPU hourly rate: unknown
Results → /content/drive/MyDrive/slm-distillation/latency_benchmarks/20261004_024956_403611
Teacher error: TypeError
Round 1/3: loading cleaned_5e5

Fetching 2 files: 100%|██████████| 2/2 [01:22<00:00, 41.12s/it]

Loading weights: 100%|██████████| 883/883 [00:36<00:00, 24.22it/s] 
/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.laye

RuntimeError: Benchmark exited with code 1. Read the traceback above. Full log: /content/drive/MyDrive/slm-distillation/latency_benchmarks/logs/20261004_024942_180912.log

In [ ]:
# ── Full training + evaluation run (~40–60 min on T4) ─────────────────────────
import os
CODE = os.environ.get('SLM_CODE_DIR', '/content/project/code')
!python "$CODE/main.py" \
    --phase 1 \
    --config "$CODE/configs/phase1_config.yaml" \
    --device_mode colab

01:46:31 | INFO     | __main__ | [main] Overriding device_mode: local_mps → colab
01:46:31 | INFO     | __main__ | [main] Colab mode: drive_root → /content/drive/MyDrive/slm-distillation
01:46:32 | INFO     | __main__ | [main] Drive already mounted — skipping drive.mount().
01:46:32 | INFO     | __main__ | [main] HF cache → /content/drive/MyDrive/slm-distillation/hf_cache
01:46:32 | INFO     | __main__ | [main] Colab Secrets loaded.
01:46:32 | INFO     | __main__ | 
  SLM Distillation — Phase 1 | Mode: train
  Config:      /content/project/code/configs/phase1_config.yaml
  Device mode: colab
  Student SLM: HuggingFaceTB/SmolLM2-1.7B-Instruct
  Teacher LLM: claude-haiku-4-5
  Drive root:  /content/drive/MyDrive/slm-distillation
01:46:32 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.
01:46:34 | INFO     | phase1.pipeline | [pipeline] Run ID : 20260914_0146_SmolLM2-1.7B-Instruct_ep3
01:46:34 | INFO     | phase1.pipeline | [pipeline] Outputs → /content/drive/MyDrive/slm-dist

In [ ]:
# ── Live demo mode ────────────────────────────────────────────────────────────
import os
CODE  = os.environ.get('SLM_CODE_DIR', '/content/project/code')
DRIVE = os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/slm-distillation')

ADAPTER = f'{DRIVE}/outputs/gemma3_5.0e-5_2epochs_2000sample_auged_3var_completion_only_loss_0.1dropout_cleaned/models/lora_adapter'  # ← UPDATE

!python "$CODE/main.py" \
    --phase 1 \
    --config "$CODE/configs/phase1_config.yaml" \
    --device_mode colab \
    --mode demo \
    --adapter_dir "$ADAPTER"

02:59:39 | INFO     | __main__ | [main] Overriding device_mode: local_mps → colab
02:59:39 | INFO     | __main__ | [main] Colab mode: drive_root → /content/drive/MyDrive/slm-distillation
02:59:39 | INFO     | __main__ | [main] Drive already mounted — skipping drive.mount().
02:59:39 | INFO     | __main__ | [main] HF cache → /content/drive/MyDrive/slm-distillation/hf_cache
02:59:39 | INFO     | __main__ | [main] Colab Secrets loaded.
02:59:39 | INFO     | __main__ | 
  SLM Distillation — Phase 1 | Mode: demo
  Config:      /content/project/code/configs/phase1_config.yaml
  Device mode: colab
  Student SLM: google/gemma-3-4b-it
  Teacher LLM: claude-haiku-4-5
  Drive root:  /content/drive/MyDrive/slm-distillation
02:59:40 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.

  SLM DISTILLATION — LIVE DEMO
  Default prompt: P1  |  Top-k: 10  |  Device: colab
  Judge mode: reference

Loading metric tools ...
02:59:51 | INFO     | datasets | TensorFlow version 2.20.0 available.
02:

In [ ]:
!git config --global user.name "Ancheng-git"
!git config --global user.email "ancheng@ad.unc.edu"

In [ ]:
# ── Push code changes to GitHub ───────────────────────────────────────────────
import subprocess, os

MSG = 'Changed SLM to HuggingFaceTB/SmolLM2-1.7B-Instruct'   # ← UPDATE
REPO = '/content/project'

for cmd in [['git','add','.'], ['git','commit','-m',MSG], ['git','push']]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
print('Done')

[BTT_SETUP_AC 8398514] Changed SLM to HuggingFaceTB/SmolLM2-1.7B-Instruct
 41 files changed, 147617 insertions(+), 38 deletions(-)
 create mode 100644 code/huggingface_tokenizers_cache/.locks/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/0ad5ecc2035b7031b88afb544ee95e2d49baa484.lock
 create mode 100644 code/huggingface_tokenizers_cache/.locks/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/207e75b189e8d8b72a8966dddcce5502f653e5ba.lock
 create mode 100644 code/huggingface_tokenizers_cache/.locks/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/44719d2e365acac0637fd25a3acf46494ca45940.lock
 create mode 100644 code/huggingface_tokenizers_cache/.locks/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/69503b13f727ba3812b6803e97442a6de05ef5eb.lock
 create mode 100644 code/huggingface_tokenizers_cache/.locks/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/8c7b22013909450429303ed10be4398bd63f5457.lock
 create mode 100644 code/huggingface_tokenizers_cache/.locks/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/da6

---
## First session only — create your branch

Only needed if your branch does not yet exist on GitHub.

In [ ]:
# ── Create branch on GitHub (run ONCE) ───────────────────────────────────────
import subprocess

MY_BRANCH = 'pair-X/model-name'   # ← CHANGE THIS

for cmd in [
    ['git', 'checkout', '-b', MY_BRANCH],
    ['git', 'push', '-u', 'origin', MY_BRANCH],
]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/project')
    print((r.stdout + r.stderr).strip())

print(f"\n✅ Branch '{MY_BRANCH}' created.")
print(f"Change BRANCH_NAME in Cell 3 to '{MY_BRANCH}' for all future sessions.")